# DD2 EoS engine — usage notebook

A guided tour of the `eos.dd2` density-dependent RMF engine. All the real work
lives in **`eos/dd2/notebook_api.py`** — every cell here just calls a
`plot_*` / `compute_*` function and shows the result, so the physics code stays
diffable and testable.

The full plot set is run **twice**: once with the published **DD2**
parametrization, then with a parametrization **built from NMPs** (`L_sym`
nudged 55 → 70 MeV). Every function takes a `Parametrization`, so the two passes
differ only in which `par` is passed in.

Physics spec: `DD2_EoS_Physics_Report.md`. Cold / T=0 unless a plot is about
temperature; nucleonic `SpeciesFlags(hyperons=False, phi_field=False)` for the
NS-structure plots.

**Setup.** Install the two packages this notebook needs straight from GitHub:
`eos` (this engine) and `metastability-nucleation` (the `add_observational_constraints`
helper used in fig 8). Run this cell first on a fresh kernel; restart the kernel
if a stale `eos` was already imported.

In [ ]:
import sys
!{sys.executable} -m pip install --no-deps --force-reinstall git+https://github.com/guerrinimirco/eos.git --quiet
print("eos package loaded successfully!")
!{sys.executable} -m pip install --no-deps --force-reinstall git+https://github.com/guerrinimirco/metastability-nucleation.git --quiet
print("nucleation package loaded successfully!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from eos.dd2 import Parametrization
from eos.dd2 import notebook_api as api

par_dd2 = Parametrization.from_dd2_defaults()

# Second-pass parametrization from nuclear-matter parameters (§NMP).
par_nmp, nmp_status = api.build_nmp_par()
print("NMP inversion:", nmp_status.message, "(ok =", nmp_status.ok, ")")
assert nmp_status.ok, "inverter did not converge — inspect nmp_status before plotting"

# TOV sequences are shared by figs 6-8; solve once per parametrization.
tov_dd2 = api.compute_tov(par_dd2, api.NUCLEONIC)
tov_nmp = api.compute_tov(par_nmp, api.NUCLEONIC)
print(f"DD2 : M_max={tov_dd2['M_max']:.3f}  R_1.4={tov_dd2['R_1p4']:.2f} km  "
      f"Lambda_1.4={tov_dd2['Lambda_1p4']:.0f}")
print(f"NMP : M_max={tov_nmp['M_max']:.3f}  R_1.4={tov_nmp['R_1p4']:.2f} km  "
      f"Lambda_1.4={tov_nmp['Lambda_1p4']:.0f}")

---
## Pass — default DD2

The entire set for `par_dd2`.

### 1. Pressure vs $n_B$ (β-equilibrium)

Neutrino-transparent npeμ matter along β-equilibrium via the warm-started `sweep_beta_eq_octet`. Log-y.

In [ ]:
api.plot_p_vs_nb(par_dd2)

### 2. Composition $Y_i$ vs $n_B$

Particle fractions along β-eq. Nucleonic here (the DD2/NMP pars carry no hyperon couplings); a DD2Y cell below shows the hyperon onsets.

In [ ]:
api.plot_composition(par_dd2)

### 3. Isentropic temperature

Temperature along constant entropy-per-baryon paths S = s/n_B (outer T-solve on the same octet kernel).

In [ ]:
api.plot_isentropic_T(par_dd2)

### 4. Speed of sound $c_s^2$

Frozen (fixed-composition) `sound_speed_adiabatic` and equilibrium `sound_speed_eq`, with the causal limit c_s²=1 marked.

In [ ]:
api.plot_sound_speed(par_dd2)

### 5. Heat capacities $C_V$, $C_P$ (T>0)

Per-baryon `heat_capacity_V` and C_P (frozen-composition Mayer relation) at fixed T = 10 MeV.

In [ ]:
api.plot_heat_capacity(par_dd2, T=10.0)

### 6. Mass-radius

Cold β-eq core + BPS crust through the repo TOV solver; M_max and R_1.4 annotated.

In [ ]:
api.plot_mass_radius(par_dd2, tov=tov_dd2)

### 7. Tidal deformability Λ-M

Λ vs mass from the same TOV sequence (`compute_tidal=True`).

In [ ]:
api.plot_lambda_mass(par_dd2, tov=tov_dd2)

### 8. M-R vs observational constraints

The M-R curve over the shipped J0030 / J0740 / HESS / GW170817 / GW190425 posteriors via `add_observational_constraints`.

In [ ]:
api.plot_mr_with_constraints(par_dd2, tov=tov_dd2)

### 9. Pressure vs $n_B$ (symmetric matter)

Symmetric nuclear matter (Y_p = 0.5) pressure, `solve_snm` sweep. Log-y (positive branch above saturation).

In [ ]:
api.plot_p_vs_nb_snm(par_dd2)

### 10. Pure neutron matter vs chiral EFT

PNM energy per particle (`solve_composition` at Y_p=0) against the chiral-EFT band.

In [ ]:
api.plot_pnm_chiral(par_dd2)

### 11. Nuclear-matter parameters

`compute_nmp` next to the DD2 reference values.

In [ ]:
print(api.format_nmp_comparison(par_dd2))

### Hyperon onsets (DD2Y)

The DD2/NMP pars are nucleonic. To see the hyperon composition, use the DD2Y
parametrization (which ships the Fortin/Marques couplings) with the full octet.

In [ ]:
par_dd2y = Parametrization.from_dd2y_defaults()
api.plot_composition(par_dd2y, flags=api.OCTET)

---
## Pass — NMP-built (L_sym 55→70)

The entire set for `par_nmp`.

### 1. Pressure vs $n_B$ (β-equilibrium)

Neutrino-transparent npeμ matter along β-equilibrium via the warm-started `sweep_beta_eq_octet`. Log-y.

In [ ]:
api.plot_p_vs_nb(par_nmp)

### 2. Composition $Y_i$ vs $n_B$

Particle fractions along β-eq. Nucleonic here (the DD2/NMP pars carry no hyperon couplings); a DD2Y cell below shows the hyperon onsets.

In [ ]:
api.plot_composition(par_nmp)

### 3. Isentropic temperature

Temperature along constant entropy-per-baryon paths S = s/n_B (outer T-solve on the same octet kernel).

In [ ]:
api.plot_isentropic_T(par_nmp)

### 4. Speed of sound $c_s^2$

Frozen (fixed-composition) `sound_speed_adiabatic` and equilibrium `sound_speed_eq`, with the causal limit c_s²=1 marked.

In [ ]:
api.plot_sound_speed(par_nmp)

### 5. Heat capacities $C_V$, $C_P$ (T>0)

Per-baryon `heat_capacity_V` and C_P (frozen-composition Mayer relation) at fixed T = 10 MeV.

In [ ]:
api.plot_heat_capacity(par_nmp, T=10.0)

### 6. Mass-radius

Cold β-eq core + BPS crust through the repo TOV solver; M_max and R_1.4 annotated.

In [ ]:
api.plot_mass_radius(par_nmp, tov=tov_nmp)

### 7. Tidal deformability Λ-M

Λ vs mass from the same TOV sequence (`compute_tidal=True`).

In [ ]:
api.plot_lambda_mass(par_nmp, tov=tov_nmp)

### 8. M-R vs observational constraints

The M-R curve over the shipped J0030 / J0740 / HESS / GW170817 / GW190425 posteriors via `add_observational_constraints`.

In [ ]:
api.plot_mr_with_constraints(par_nmp, tov=tov_nmp)

### 9. Pressure vs $n_B$ (symmetric matter)

Symmetric nuclear matter (Y_p = 0.5) pressure, `solve_snm` sweep. Log-y (positive branch above saturation).

In [ ]:
api.plot_p_vs_nb_snm(par_nmp)

### 10. Pure neutron matter vs chiral EFT

PNM energy per particle (`solve_composition` at Y_p=0) against the chiral-EFT band.

In [ ]:
api.plot_pnm_chiral(par_nmp)

### 11. Nuclear-matter parameters

`compute_nmp` next to the DD2 reference values.

In [ ]:
print(api.format_nmp_comparison(par_nmp))

---
## Speed test — SFHo vs DD2

Matched β-equilibrium density sweep: the DD2 fast path
(`sweep_beta_eq_octet(..., analytic_jac=True)`, first call discarded for Numba
compile) against the SFHo table generator. Anchor: a 100-pt nucleonic DD2 sweep
is ~35 ms (~0.35 ms/pt) on the dev machine; treat these as sanity values, not
targets.

In [ ]:
bench = api.benchmark_dd2_vs_sfho(par_dd2)
print(f"grid points : {bench['n_points']}")
print(f"DD2  (fast) : {bench['dd2_ms_per_pt']:.3f} ms/pt")
print(f"SFHo        : {bench['sfho_ms_per_pt']:.3f} ms/pt")
print(f"ratio SFHo/DD2 : {bench['ratio']:.2f}")